In [1]:
using LowLevelFEM, LinearAlgebra
using IterativeSolvers, Preconditioners

openGeometry("torus.geo")

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 2 for different dependency version already loaded, 6 for different Julia build configuration, 1 for file size changed)
Precompiling packages...
  14.7 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 16 seconds. 134 already precompiled.
[ Info: Precompiling LowLevelFEMIterativeSolversExt [55978d5f-b2ed-541f-a526-69ba3ceeb2d1] (caches not reused: 1 for different dependency version already loaded, 4 for different Julia build configuration)
Precompiling packages...
   2.6 s  ✓ LowLevelFEM → LowLevelFEMIterativeSolversExt
  1 dependency successfully precompiled in 6 seconds. 137 already precompiled.
[ Info: Precompiling LowLevelFEMPreconditionersExt [731aa7ef-afda-5ca4-bfd0-d21a0237399b] (caches not reused: 1 for different dependency version already loaded, 4 for different Julia build configuration)
Precompiling packages...
   3.2 s  ✓ LowLevelFEM → LowLevelFEMPreconditionersExt
  1 depen

In [2]:
mat = material("body")
prob = Problem([mat])
U = Field([mat], type=:VectorField, dim=3, field=:u)

Problem("torus", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 26075, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing))

In [3]:
ndofs(U)

78225

In [4]:
@time K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U));

  7.595675 seconds (9.76 M allocations: 817.692 MiB, 5.80% gc time, 147.70% compilation time)


In [5]:
f = ∫(U ⋅ [0,0, -5000/78.54], Γ="load");

In [6]:
supp = BoundaryCondition("supp", ux=0, uy=0, uz=0);

In [7]:
showDoFResults(f, name="terhelés");

In [8]:
@time u = solveField(Symmetric(K), f, support=[supp], solver=:cg)
showDoFResults(u, name="elmozdulás");

 34.059982 seconds (25.95 M allocations: 1.918 GiB, 2.05% gc time, 70.01% compilation time)


In [9]:
ε = (u ∘ ∇ + ∇ ∘ u) / 2

E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1-2ν) * trace(ε) * I);

In [10]:
n_mid = normalVector(prob, "mid")
σ_mid = elementsToElements(σ, onPhysicalGroup="mid", fromPhysicalGroup="body1")

ρn = σ_mid * n_mid
σn = n_mid * (ρn ⋅ n_mid)
τn = ρn - σn;

In [11]:
showElementResults(ρn, name="ρₙ")
showElementResults(n_mid, name="normál vektor")
showElementResults(σn, name="σₙ")
showElementResults(τn, name="τₙ")
sn = showDoFResults(ρn * n_mid, name="sn")
tn = showDoFResults(τn, name="tn")
plotOnPath("path", sn, name="normál feszültség")
plotOnPath("path", tn, name="csúsztató feszültség")

9

In [12]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
